# RSNA Knee — producción

Pipeline: DICOM → 2.5D → DenseNet121 → Attention → 12 logits.

Código en `rsna_knee/`. Este notebook solo lo ejecuta.


## 1. Entorno


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

here = Path.cwd().resolve()
for cand in (here, here / "produccion", here.parent):
    if (cand / "rsna_knee").is_dir():
        sys.path.insert(0, str(cand))
        break

from rsna_knee import KneePipeline, cargar_estudio, cargar_tablas, device

dev = device()
print("device:", dev)


## 2. Datos


In [ ]:
train_df, series_df, LABELS, TRAIN_IMAGES, COMP = cargar_tablas()
print("comp:", COMP)
print("imágenes:", TRAIN_IMAGES, TRAIN_IMAGES.exists())
print("train:", train_df.shape, "series:", series_df.shape)
print("labels:", LABELS)
assert TRAIN_IMAGES.exists()
assert len(LABELS) == 12
display(train_df.head(1))


## 3. Un estudio → 12 probabilidades


In [ ]:
pipe = KneePipeline(n_labels=len(LABELS)).to(dev)
pipe.eval()

study_id = str(train_df.iloc[0]["StudyInstanceUID"])
estudio = cargar_estudio(study_id, series_df, TRAIN_IMAGES)
for plano, series in estudio.items():
    print(plano, len(series), "series")

with torch.no_grad():
    logits = pipe(estudio)
    probs = torch.sigmoid(logits).cpu().numpy()

print("logits:", tuple(logits.shape))
display(pd.DataFrame({"Etiqueta": LABELS, "Probabilidad": probs}))
print("Pesos ImageNet + cabeza sin entrenar: las probs no son diagnósticas.")
